In [1]:
import time
import boto3
import sagemaker
import json
import numpy as np
from sagemaker import get_execution_role
from sagemaker.amazon.amazon_estimator import get_image_uri
from sagemaker.tuner import IntegerParameter, CategoricalParameter, ContinuousParameter, HyperparameterTuner
from sagemaker.model import Model
from sagemaker.predictor import RealTimePredictor

# Configuring the built-in Image Classification algorithm

## Create a unique job name 

In [2]:
job_name_prefix = 'breast-cancer-detection'
timestamp = time.strftime('-%Y-%m-%d-%H-%M-%S', time.gmtime())
job_name = job_name_prefix + timestamp

## Specify the input and output paths for the job

In [3]:
bucket = 'alexbc.datalake.us-east-2'
input_prefix = 'breast-cancer-histopathology/curated-data/recordio'
input_train = 's3://{}/{}/train/'.format(bucket, input_prefix)
input_test = 's3://{}/{}/test/'.format(bucket, input_prefix)

output_prefix = 'breast-cancer-histopathology/mmodel-output/sagemakerapi'
output_path = 's3://{}/{}/'.format(bucket, output_prefix)

## Creating an Estimator to build image classification

In [4]:
role = get_execution_role()
training_image = get_image_uri(boto3.Session().region_name, 'image-classification')

sagemaker_session = sagemaker.Session()
estimator = sagemaker.estimator.Estimator(training_image, 
                                          role, 
                                          train_instance_count=1,
                                          train_instance_type='ml.p2.xlarge',
                                          train_volume_size=50,
                                          train_max_run=360000,
                                          output_path=output_path, 
                                          sagemaker_session=sagemaker_session,
                                          input_mode='Pipe')

In [5]:
estimator.set_hyperparameters(num_classes=2,
                              num_training_samples=194266,
                              num_layers=18,
                              mini_batch_size=128,
                              image_shape='3,50,50',
                              augmentation_type='crop_color_transform',
                              epochs=5,
                              learning_rate=0.01,
                              use_pretrained_model=1)

# Create a training job

In [6]:
s3_input_train = sagemaker.s3_input(s3_data=input_train, content_type='application/x-recordio')
s3_input_validation = sagemaker.s3_input(s3_data=input_test, content_type='application/x-recordio')

In [ ]:
estimator.fit({
    'train': s3_input_train,
    'validation': s3_input_validation
}, job_name=job_name)

2020-01-31 20:10:06 Starting - Starting the training job...
2020-01-31 20:10:08 Starting - Launching requested ML instances...
2020-01-31 20:11:04 Starting - Preparing the instances for training............
2020-01-31 20:12:44 Downloading - Downloading input data...
2020-01-31 20:13:12 Training - Downloading the training image.....Docker entrypoint called with argument(s): train
[01/31/2020 20:14:19 INFO 139717033891648] Reading default configuration from /opt/amazon/lib/python2.7/site-packages/image_classification/default-input.json: {u'beta_1': 0.9, u'gamma': 0.9, u'beta_2': 0.999, u'optimizer': u'sgd', u'use_pretrained_model': 0, u'eps': 1e-08, u'epochs': 30, u'lr_scheduler_factor': 0.1, u'num_layers': 152, u'image_shape': u'3,224,224', u'precision_dtype': u'float32', u'mini_batch_size': 32, u'weight_decay': 0.0001, u'learning_rate': 0.1, u'momentum': 0}
[01/31/2020 20:14:19 INFO 139717033891648] Merging with provided configuration from /opt/ml/input/config/hyperparameters.json: {u'

[01/31/2020 20:15:33 INFO 139717033891648] Epoch[0] Batch [700]#011Speed: 1315.781 samples/sec#011accuracy=0.832204
[01/31/2020 20:15:34 INFO 139717033891648] Epoch[0] Batch [720]#011Speed: 1316.903 samples/sec#011accuracy=0.832524
[01/31/2020 20:15:36 INFO 139717033891648] Epoch[0] Batch [740]#011Speed: 1318.211 samples/sec#011accuracy=0.832975
[01/31/2020 20:15:38 INFO 139717033891648] Epoch[0] Batch [760]#011Speed: 1319.061 samples/sec#011accuracy=0.833320
[01/31/2020 20:15:40 INFO 139717033891648] Epoch[0] Batch [780]#011Speed: 1319.713 samples/sec#011accuracy=0.833707
[01/31/2020 20:15:42 INFO 139717033891648] Epoch[0] Batch [800]#011Speed: 1320.682 samples/sec#011accuracy=0.834075
[01/31/2020 20:15:44 INFO 139717033891648] Epoch[0] Batch [820]#011Speed: 1321.532 samples/sec#011accuracy=0.834206
[01/31/2020 20:15:46 INFO 139717033891648] Epoch[0] Batch [840]#011Speed: 1322.452 samples/sec#011accuracy=0.834256
[01/31/2020 20:15:48 INFO 139717033891648] Epoch[0] Batch [860]#011Speed

[01/31/2020 20:18:08 INFO 139717033891648] Epoch[1] Batch [520]#011Speed: 1336.930 samples/sec#011accuracy=0.834108
[01/31/2020 20:18:10 INFO 139717033891648] Epoch[1] Batch [540]#011Speed: 1336.608 samples/sec#011accuracy=0.834104
[01/31/2020 20:18:12 INFO 139717033891648] Epoch[1] Batch [560]#011Speed: 1337.033 samples/sec#011accuracy=0.834642
[01/31/2020 20:19:40 INFO 139717033891648] Epoch[1] Batch [1480]#011Speed: 1342.229 samples/sec#011accuracy=0.843877
[01/31/2020 20:19:42 INFO 139717033891648] Epoch[1] Batch [1500]#011Speed: 1342.235 samples/sec#011accuracy=0.843812
[01/31/2020 20:19:43 INFO 139717033891648] Epoch[1] Train-accuracy=0.843946
[01/31/2020 20:19:43 INFO 139717033891648] Epoch[1] Time cost=144.558
[01/31/2020 20:20:11 INFO 139717033891648] Epoch[1] Validation-accuracy=0.841386
[01/31/2020 20:20:11 INFO 139717033891648] Storing the best model with validation accuracy: 0.841386
[01/31/2020 20:20:11 INFO 139717033891648] Saved checkpoint to "/opt/ml/model/image-classi

[01/31/2020 20:22:07 INFO 139717033891648] Epoch[2] Batch [1200]#011Speed: 1331.406 samples/sec#011accuracy=0.832379
[01/31/2020 20:22:09 INFO 139717033891648] Epoch[2] Batch [1220]#011Speed: 1331.496 samples/sec#011accuracy=0.832130
[01/31/2020 20:22:11 INFO 139717033891648] Epoch[2] Batch [1240]#011Speed: 1331.895 samples/sec#011accuracy=0.831405
[01/31/2020 20:22:12 INFO 139717033891648] Epoch[2] Batch [1260]#011Speed: 1332.109 samples/sec#011accuracy=0.831291
[01/31/2020 20:22:14 INFO 139717033891648] Epoch[2] Batch [1280]#011Speed: 1332.155 samples/sec#011accuracy=0.831083
[01/31/2020 20:22:16 INFO 139717033891648] Epoch[2] Batch [1300]#011Speed: 1332.504 samples/sec#011accuracy=0.830623
[01/31/2020 20:22:18 INFO 139717033891648] Epoch[2] Batch [1320]#011Speed: 1332.497 samples/sec#011accuracy=0.830390
[01/31/2020 20:22:20 INFO 139717033891648] Epoch[2] Batch [1340]#011Speed: 1332.881 samples/sec#011accuracy=0.830333
[01/31/2020 20:22:22 INFO 139717033891648] Epoch[2] Batch [1360]

[01/31/2020 20:24:41 INFO 139717033891648] Epoch[3] Batch [1000]#011Speed: 1338.738 samples/sec#011accuracy=0.828687
[01/31/2020 20:24:43 INFO 139717033891648] Epoch[3] Batch [1020]#011Speed: 1338.918 samples/sec#011accuracy=0.828531
[01/31/2020 20:24:45 INFO 139717033891648] Epoch[3] Batch [1040]#011Speed: 1338.995 samples/sec#011accuracy=0.828560
[01/31/2020 20:24:47 INFO 139717033891648] Epoch[3] Batch [1060]#011Speed: 1338.577 samples/sec#011accuracy=0.828876
[01/31/2020 20:24:49 INFO 139717033891648] Epoch[3] Batch [1080]#011Speed: 1338.218 samples/sec#011accuracy=0.829065
[01/31/2020 20:24:50 INFO 139717033891648] Epoch[3] Batch [1100]#011Speed: 1338.195 samples/sec#011accuracy=0.829147
[01/31/2020 20:24:52 INFO 139717033891648] Epoch[3] Batch [1120]#011Speed: 1338.314 samples/sec#011accuracy=0.829568
[01/31/2020 20:24:54 INFO 139717033891648] Epoch[3] Batch [1140]#011Speed: 1338.308 samples/sec#011accuracy=0.829549
[01/31/2020 20:24:56 INFO 139717033891648] Epoch[3] Batch [1160]

[01/31/2020 20:27:13 INFO 139717033891648] Epoch[4] Batch [780]#011Speed: 1333.796 samples/sec#011accuracy=0.845260
[01/31/2020 20:27:15 INFO 139717033891648] Epoch[4] Batch [800]#011Speed: 1334.096 samples/sec#011accuracy=0.845252
[01/31/2020 20:27:17 INFO 139717033891648] Epoch[4] Batch [820]#011Speed: 1334.225 samples/sec#011accuracy=0.845016
[01/31/2020 20:27:19 INFO 139717033891648] Epoch[4] Batch [840]#011Speed: 1334.225 samples/sec#011accuracy=0.844976
[01/31/2020 20:27:21 INFO 139717033891648] Epoch[4] Batch [860]#011Speed: 1334.405 samples/sec#011accuracy=0.844975
[01/31/2020 20:27:22 INFO 139717033891648] Epoch[4] Batch [880]#011Speed: 1334.458 samples/sec#011accuracy=0.845045
[01/31/2020 20:27:24 INFO 139717033891648] Epoch[4] Batch [900]#011Speed: 1334.742 samples/sec#011accuracy=0.844825
[01/31/2020 20:27:26 INFO 139717033891648] Epoch[4] Batch [920]#011Speed: 1335.123 samples/sec#011accuracy=0.844988
[01/31/2020 20:27:28 INFO 139717033891648] Epoch[4] Batch [940]#011Speed

# Deploying the best model found by the tuning job

## Get the execution role and the hosting image URI for Image Classification

In [ ]:
role = get_execution_role()
hosting_image = get_image_uri(boto3.Session().region_name, 'image-classification')

## Configure hosting instances

In [ ]:
instance_count = 1
instance_type = 'ml.m4.xlarge'

## Create a unique model name

In [ ]:
model_name_prefix = 'bcd-image-classification-high-level'
timestamp = time.strftime('-%Y-%m-%d-%H-%M-%S', time.gmtime())
model_name = model_name_prefix + timestamp

## Create a Model object

In [ ]:
model_artifacts_s3_path = 's3://sagemaker-data-jv/breast-cancer-detection/output/bcd-tuning-2019-06-17-12-24-43-001-f302e721/output/model.tar.gz'
model = Model(
    name=model_name,
    model_data=model_artifacts_s3_path,
    image=hosting_image,
    role=role,
    predictor_cls=lambda endpoint_name, sagemaker_session: RealTimePredictor(endpoint_name, sagemaker_session)
)

## Create a unique endpoint name

In [ ]:
endpoint_name_prefix = 'breast-cancer-detection-ep'
timestamp = time.strftime('-%Y-%m-%d-%H-%M-%S', time.gmtime())
endpoint_name = endpoint_name_prefix + timestamp

## Create a model, an endpoint configuration and an endpoint

In [ ]:
predictor = model.deploy(
    endpoint_name=endpoint_name,
    initial_instance_count=instance_count,
    instance_type=instance_type
)

# Testing the deployed model

In [ ]:
def predict_breast_cancer(image_path):
    with open(image_path, 'rb') as f:
        payload = f.read()
        payload = bytearray(payload)
    response = predictor.predict(payload)
    result = json.loads(response)
    print('Probabilities for all classes: ', result)
    predicted_class = np.argmax(result)
    if predicted_class == 0:
        print('Breast cancer not detected')
    else:
        print('Breast cancer detected')

In [ ]:
predict_breast_cancer('images/0/8975_idx5_x2851_y1201_class0.png')

In [ ]:
predict_breast_cancer('images/1/10253_idx5_x551_y651_class1.png')

# Deleting endpoint

In [ ]:
sagemaker.Session().delete_endpoint(predictor.endpoint)